# 04_ingest_clean_aisstream.ipynb

This notebook ingests **AIS vessel movement data** from **AISStream** via WebSocket, stores the raw stream, and creates a cleaned CSV for the project pipeline.

## What this notebook does
1. Installs the required libraries
2. Sets repo paths for `data/raw` and `data/clean`
3. Lets you paste your **AISStream API key**
4. Subscribes to selected port bounding boxes
5. Collects AIS messages for a fixed duration
6. Saves raw JSONL and cleaned CSV outputs

## Expected outputs
- `data/raw/aisstream_raw.jsonl`
- `data/clean/aisstream_clean.csv`

## Notes
- AISStream is a **WebSocket** source, not a normal REST API.
- This notebook is designed for **Google Colab** + **Google Drive**.
- Keep your API key private if you later upload the repo anywhere public.


In [1]:
# Install libraries
!pip install websockets nest_asyncio pandas -q


In [2]:
import os, sys

# ── Environment detection: works in Colab and local/GitHub clone ──────────
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/repo'
    print('Running in Google Colab')
except ImportError:
    BASE = os.path.abspath(os.path.join(os.getcwd(), '..'))
    print(f'Running locally — BASE: {BASE}')

RAW_DIR        = os.path.join(BASE, 'data', 'raw')
CLEAN_DIR      = os.path.join(BASE, 'data', 'clean')
RAW_JSONL_PATH = os.path.join(RAW_DIR,   'aisstream_raw.jsonl')
CLEAN_CSV_PATH = os.path.join(CLEAN_DIR, 'aisstream_clean.csv')

os.makedirs(RAW_DIR,   exist_ok=True)
os.makedirs(CLEAN_DIR, exist_ok=True)

print(f'BASE: {BASE}')
print(f'Raw JSONL: {RAW_JSONL_PATH}')
print(f'Clean CSV: {CLEAN_CSV_PATH}')


Mounted at /content/drive
BASE exists: True
RAW_DIR exists: True
CLEAN_DIR exists: True
Raw output: /content/drive/MyDrive/repo/data/raw/aisstream_raw.jsonl
Clean output: /content/drive/MyDrive/repo/data/clean/aisstream_clean.csv


In [3]:
# Imports
import asyncio
import json
from datetime import datetime, timezone

import nest_asyncio
import pandas as pd
import websockets

nest_asyncio.apply()
pd.set_option('display.max_columns', 200)


## 1) Add your AISStream API key

Paste your API key below.  
Do not share the notebook with the real key visible.


In [4]:
AISSTREAM_API_KEY = 'PASTE_YOUR_AISSTREAM_API_KEY_HERE'

if AISSTREAM_API_KEY == 'PASTE_YOUR_AISSTREAM_API_KEY_HERE':
    print('⚠️ Please paste your AISStream API key before running the stream cell.')
else:
    print('API key added.')


API key added.


## 2) Choose the ports / areas to monitor

Each port has a small bounding box:  
`[[lat1, lon1], [lat2, lon2]]`

You can keep these defaults or edit them.


In [5]:
# Example port bounding boxes
# PORT_BBOXES = {
#     'Singapore': [[1.20, 103.60], [1.40, 104.05]],
#     'Shanghai': [[31.05, 121.20], [31.45, 122.10]],
#     'Rotterdam': [[51.80, 3.80], [52.10, 4.70]],
#     'Los_Angeles_Long_Beach': [[33.60, -118.35], [33.85, -117.95]],
#     'Jebel_Ali': [[24.90, 54.95], [25.15, 55.15]],
#     'Beirut': [[33.85, 35.45], [33.95, 35.60]],
# }

# SELECTED_PORTS = ['Singapore', 'Shanghai', 'Rotterdam', 'Los_Angeles_Long_Beach', 'Jebel_Ali', 'Beirut']
# BOUNDING_BOXES = [PORT_BBOXES[p] for p in SELECTED_PORTS]

# print('Selected ports:', SELECTED_PORTS)
# print('Bounding boxes to subscribe:', BOUNDING_BOXES)

PORT_BBOXES = {
    'Singapore': [[1.00, 103.40], [1.60, 104.40]],
    'Shanghai':  [[30.80, 120.80], [31.80, 122.40]],
}

SELECTED_PORTS = ['Singapore', 'Shanghai']
BOUNDING_BOXES = [PORT_BBOXES[p] for p in SELECTED_PORTS]

print('Selected ports:', SELECTED_PORTS)
print('Bounding boxes to subscribe:', BOUNDING_BOXES)

Selected ports: ['Singapore', 'Shanghai']
Bounding boxes to subscribe: [[[1.0, 103.4], [1.6, 104.4]], [[30.8, 120.8], [31.8, 122.4]]]


## 3) Stream capture settings

- `RUN_SECONDS`: how long to listen
- `MAX_MESSAGES`: hard cap on collected messages
- `FILTER_MESSAGE_TYPES`: keep only the AIS message types you want

For a class project, start with a small run first.


In [6]:
RUN_SECONDS = 300
MAX_MESSAGES = 2000
FILTER_MESSAGE_TYPES = None

print('Run seconds:', RUN_SECONDS)
print('Max messages:', MAX_MESSAGES)
print('Message types:', FILTER_MESSAGE_TYPES)


Run seconds: 300
Max messages: 2000
Message types: None


In [7]:
# Helpers
def point_in_bbox(lat, lon, bbox):
    (lat1, lon1), (lat2, lon2) = bbox
    min_lat, max_lat = min(lat1, lat2), max(lat1, lat2)
    min_lon, max_lon = min(lon1, lon2), max(lon1, lon2)
    return (min_lat <= lat <= max_lat) and (min_lon <= lon <= max_lon)

def guess_port_from_point(lat, lon, port_bboxes):
    if lat is None or lon is None or pd.isna(lat) or pd.isna(lon):
        return None
    for port_name, bbox in port_bboxes.items():
        if point_in_bbox(float(lat), float(lon), bbox):
            return port_name
    return None


## 4) Capture raw AIS messages from AISStream

This cell:
- opens a WebSocket to AISStream
- sends the subscription JSON
- listens for messages
- saves the raw stream as **JSONL**

If the service is quiet for the selected boxes, you may get fewer rows.


In [8]:
async def capture_aisstream(
    api_key,
    bounding_boxes,
    filter_message_types=None,
    run_seconds=60,
    max_messages=500,
    raw_jsonl_path=RAW_JSONL_PATH,
):
    ws_url = 'wss://stream.aisstream.io/v0/stream'
    subscription_message = {
        'APIKey': api_key,
        'BoundingBoxes': bounding_boxes,
    }

    if filter_message_types:
        subscription_message['FilterMessageTypes'] = filter_message_types

    messages = []
    start_utc = datetime.now(timezone.utc)

    print('Connecting to AISStream...')
    async with websockets.connect(ws_url, ping_interval=20, ping_timeout=20) as websocket:
        await websocket.send(json.dumps(subscription_message))
        print('Subscription sent.')
        print('Listening...')

        while True:
            elapsed = (datetime.now(timezone.utc) - start_utc).total_seconds()
            if elapsed >= run_seconds:
                print('Stopped because run time limit was reached.')
                break
            if len(messages) >= max_messages:
                print('Stopped because max message limit was reached.')
                break

            timeout_left = max(1, run_seconds - int(elapsed))
            try:
                message_json = await asyncio.wait_for(websocket.recv(), timeout=timeout_left)
                message = json.loads(message_json)
                messages.append(message)
            except asyncio.TimeoutError:
                print('No more messages before timeout window ended.')
                break

    with open(raw_jsonl_path, 'w', encoding='utf-8') as f:
        for msg in messages:
            f.write(json.dumps(msg, ensure_ascii=False) + '\n')

    print(f'Raw messages captured: {len(messages):,}')
    print(f'Raw JSONL saved to: {raw_jsonl_path}')
    return messages

if AISSTREAM_API_KEY == 'PASTE_YOUR_AISSTREAM_API_KEY_HERE':
    print('⚠️ Paste your AISStream API key first.')
else:
    raw_messages = asyncio.get_event_loop().run_until_complete(
        capture_aisstream(
            api_key=AISSTREAM_API_KEY,
            bounding_boxes=BOUNDING_BOXES,
            filter_message_types=FILTER_MESSAGE_TYPES,
            run_seconds=RUN_SECONDS,
            max_messages=MAX_MESSAGES,
            raw_jsonl_path=RAW_JSONL_PATH,
        )
    )


Connecting to AISStream...
Subscription sent.
Listening...
Stopped because run time limit was reached.
Raw messages captured: 998
Raw JSONL saved to: /content/drive/MyDrive/repo/data/raw/aisstream_raw.jsonl


## 5) Normalize raw messages into a table

This step extracts useful fields such as:
- message type
- MMSI / ship ID
- latitude / longitude
- speed / heading when available
- ship name and IMO when available
- guessed port from the selected bounding boxes


In [9]:
def flatten_ais_messages(messages, port_bboxes):
    rows = []

    for msg in messages:
        msg_type = msg.get('MessageType')
        metadata = msg.get('MetaData') or msg.get('Metadata') or {}
        message_block = msg.get('Message') or {}
        payload = message_block.get(msg_type, {}) if isinstance(message_block, dict) else {}

        lat = metadata.get('latitude', metadata.get('Latitude', payload.get('Latitude')))
        lon = metadata.get('longitude', metadata.get('Longitude', payload.get('Longitude')))
        mmsi = payload.get('UserID', metadata.get('MMSI', metadata.get('UserID')))
        ship_name = metadata.get('ShipName', payload.get('Name'))
        imo = metadata.get('IMO', payload.get('ImoNumber'))
        nav_status = payload.get('NavigationalStatus')
        sog = payload.get('Sog', payload.get('SpeedOverGround'))
        cog = payload.get('Cog', payload.get('CourseOverGround'))
        true_heading = payload.get('TrueHeading')
        destination = payload.get('Destination')
        eta = payload.get('Eta')
        call_sign = payload.get('CallSign')
        vessel_type = payload.get('Type', payload.get('TypeOfShipAndCargo'))
        source_ts = metadata.get('time_utc', metadata.get('time', metadata.get('Timestamp')))
        raw_message_timestamp = payload.get('Timestamp')

        rows.append({
            'message_type': msg_type,
            'mmsi': mmsi,
            'ship_name': ship_name,
            'imo': imo,
            'latitude': lat,
            'longitude': lon,
            'navigational_status': nav_status,
            'sog_knots': sog,
            'cog_degrees': cog,
            'true_heading': true_heading,
            'destination': destination,
            'eta': eta,
            'call_sign': call_sign,
            'vessel_type': vessel_type,
            'source_timestamp': source_ts,
            'ais_timestamp_field': raw_message_timestamp,
            'port_guess': guess_port_from_point(lat, lon, port_bboxes),
            'source': 'aisstream',
            'data_type': 'vessel_movement',
            'granularity': 'event',
        })

    return pd.DataFrame(rows)

if 'raw_messages' not in globals():
    raw_messages = []
    if os.path.exists(RAW_JSONL_PATH):
        with open(RAW_JSONL_PATH, 'r', encoding='utf-8') as f:
            raw_messages = [json.loads(line) for line in f if line.strip()]

df = flatten_ais_messages(raw_messages, PORT_BBOXES)

print('Rows:', len(df))
print('Columns:', df.shape[1])
df.head(5)


Rows: 998
Columns: 20


,message_type,mmsi,ship_name,imo,latitude,longitude,navigational_status,sog_knots,cog_degrees,true_heading,destination,eta,call_sign,vessel_type,source_timestamp,ais_timestamp_field,port_guess,source,data_type,granularity
0,StaticDataReport,525100095,MEGAMAS SWEET,NaN,1.26393,103.83230,NaN,NaN,NaN,NaN,None,None,None,NaN,2026-04-04 21:09:42.257585082 +0000 UTC,NaN,Singapore,aisstream,vessel_movement,event
1,PositionReport,636013118,YM INSTRUCTION,NaN,1.21927,103.81743,0.0,1.8,240.5,270.0,None,None,None,NaN,2026-04-04 21:09:42.608997997 +0000 UTC,42.0,Singapore,aisstream,vessel_movement,event
2,PositionReport,566753000,HAI SOON 33,NaN,1.20711,103.85619,0.0,7.9,64.6,52.0,None,None,None,NaN,2026-04-04 21:09:42.633423367 +0000 UTC,43.0,Singapore,aisstream,vessel_movement,event
3,PositionReport,538011746,BLUE GRETHA,NaN,1.22248,103.90738,0.0,5.0,248.1,247.0,None,None,None,NaN,2026-04-04 21:09:42.791314659 +0000 UTC,43.0,Singapore,aisstream,vessel_movement,event
4,PositionReport,563074570,VISION 225,NaN,1.26290,103.82865,15.0,0.0,228.6,511.0,None,None,None,NaN,2026-04-04 21:09:43.13851194 +0000 UTC,40.0,Singapore,aisstream,vessel_movement,event


## 6) Clean and standardize


In [10]:
if 'mmsi' in df.columns:
    df['mmsi'] = pd.to_numeric(df['mmsi'], errors='coerce').astype('Int64')

for col in ['imo', 'latitude', 'longitude', 'sog_knots', 'cog_degrees', 'true_heading']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# text_cols = [
#     'message_type', 'ship_name', 'destination', 'call_sign',
#     'vessel_type', 'port_guess', 'source', 'data_type', 'granularity'
# ]
text_cols = [
    'message_type', 'ship_name', 'destination', 'call_sign',
    'port_guess', 'source', 'data_type', 'granularity'
]

for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype('string').str.strip()

for col in ['source_timestamp']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)

capture_utc = pd.Timestamp.utcnow()
df['capture_utc'] = capture_utc

if 'latitude' in df.columns:
    df.loc[~df['latitude'].between(-90, 90, inclusive='both'), 'latitude'] = pd.NA
if 'longitude' in df.columns:
    df.loc[~df['longitude'].between(-180, 180, inclusive='both'), 'longitude'] = pd.NA

# Before removing duplicates, check for any columns that might contain unhashable types (like dictionaries)
# and convert them to a string representation to avoid TypeError.
for col in df.columns:
    # Check if the column is of 'object' dtype (where dictionaries would reside)
    # and if it contains any actual dictionary values.
    if df[col].dtype == 'object' and df[col].apply(lambda x: isinstance(x, dict)).any():
        print(f"Detected unhashable 'dict' type in column '{col}'. Converting to string for `drop_duplicates`.")
        df[col] = df[col].astype(str)

before = len(df)
df = df.drop_duplicates().copy()
print('Exact duplicates removed:', before - len(df))

if 'source_timestamp' in df.columns:
    ts_base = df['source_timestamp'].fillna(df['capture_utc'])
else:
    ts_base = df['capture_utc']

df['event_time_utc'] = ts_base
df['year'] = ts_base.dt.year.astype('Int64')
df['month'] = ts_base.dt.month.astype('Int64')
df['day'] = ts_base.dt.day.astype('Int64')
df['hour'] = ts_base.dt.hour.astype('Int64')

print('Cleaning complete')
print(df.dtypes)
df.head(5)


Detected unhashable 'dict' type in column 'eta'. Converting to string for `drop_duplicates`.
Exact duplicates removed: 49
Cleaning complete
message_type                string[python]
mmsi                                 Int64
ship_name                   string[python]
imo                                float64
latitude                           float64
longitude                          float64
navigational_status                float64
sog_knots                          float64
cog_degrees                        float64
true_heading                       float64
destination                 string[python]
eta                                 object
call_sign                   string[python]
vessel_type                        float64
source_timestamp       datetime64[ns, UTC]
ais_timestamp_field                float64
port_guess                  string[python]
source                      string[python]
data_type                   string[python]
granularity                 string[python]


/tmp/ipykernel_4224/593053151.py:23: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df[col] = pd.to_datetime(df[col], errors='coerce', utc=True)


,message_type,mmsi,ship_name,imo,latitude,longitude,navigational_status,sog_knots,cog_degrees,true_heading,destination,eta,call_sign,vessel_type,source_timestamp,ais_timestamp_field,port_guess,source,data_type,granularity,capture_utc,event_time_utc,year,month,day,hour
0,StaticDataReport,525100095,MEGAMAS SWEET,NaN,1.26393,103.83230,NaN,NaN,NaN,NaN,<NA>,None,<NA>,NaN,NaT,NaN,Singapore,aisstream,vessel_movement,event,2026-04-04 21:16:19.891862+00:00,2026-04-04 21:16:19.891862+00:00,2026,4,4,21
1,PositionReport,636013118,YM INSTRUCTION,NaN,1.21927,103.81743,0.0,1.8,240.5,270.0,<NA>,None,<NA>,NaN,NaT,42.0,Singapore,aisstream,vessel_movement,event,2026-04-04 21:16:19.891862+00:00,2026-04-04 21:16:19.891862+00:00,2026,4,4,21
2,PositionReport,566753000,HAI SOON 33,NaN,1.20711,103.85619,0.0,7.9,64.6,52.0,<NA>,None,<NA>,NaN,NaT,43.0,Singapore,aisstream,vessel_movement,event,2026-04-04 21:16:19.891862+00:00,2026-04-04 21:16:19.891862+00:00,2026,4,4,21
3,PositionReport,538011746,BLUE GRETHA,NaN,1.22248,103.90738,0.0,5.0,248.1,247.0,<NA>,None,<NA>,NaN,NaT,43.0,Singapore,aisstream,vessel_movement,event,2026-04-04 21:16:19.891862+00:00,2026-04-04 21:16:19.891862+00:00,2026,4,4,21
4,PositionReport,563074570,VISION 225,NaN,1.26290,103.82865,15.0,0.0,228.6,511.0,<NA>,None,<NA>,NaN,NaT,40.0,Singapore,aisstream,vessel_movement,event,2026-04-04 21:16:19.891862+00:00,2026-04-04 21:16:19.891862+00:00,2026,4,4,21


In [11]:
#added cell
# Decode AIS codes + add derived columns

# Make sure these stay numeric
if 'vessel_type' in df.columns:
    df['vessel_type'] = pd.to_numeric(df['vessel_type'], errors='coerce')

if 'navigational_status' in df.columns:
    df['navigational_status'] = pd.to_numeric(df['navigational_status'], errors='coerce')

# Vessel type decoding
VESSEL_TYPE_MAP = {
    0:  'Not available',
    9:  'Helicopter',
    13: 'SAR aircraft',
    20: 'Wing in ground',
    24: 'Pilot vessel',
    25: 'SAR vessel',
    26: 'Tug',
    27: 'Port tender',
    28: 'Vessel (anti-pollution)',
    29: 'Law enforcement',
    30: 'Fishing vessel',
    31: 'Towing vessel',
    32: 'Towing (long/wide)',
    33: 'Dredger',
    34: 'Dive ops vessel',
    35: 'Military vessel',
    36: 'Sailing vessel',
    37: 'Pleasure craft',
    40: 'High-speed craft',
    41: 'High-speed craft',
    50: 'Pilot vessel',
    51: 'SAR vessel',
    52: 'Tug',
    53: 'Port tender',
    54: 'Anti-pollution vessel',
    55: 'Law enforcement',
    57: 'Medical transport',
    58: 'Non-combatant ship',
    59: 'Passenger ferry',
    60: 'Passenger ship',
    61: 'Passenger ship',
    62: 'Passenger ship',
    63: 'Passenger ship',
    69: 'Passenger ship (other)',
    70: 'Cargo vessel',
    71: 'Cargo vessel (hazmat A)',
    72: 'Cargo vessel (hazmat B)',
    73: 'Cargo vessel (hazmat C)',
    74: 'Cargo vessel (hazmat D)',
    77: 'Cargo vessel',
    79: 'Cargo vessel (other)',
    80: 'Tanker',
    81: 'Tanker (hazmat A)',
    82: 'Tanker (hazmat B)',
    83: 'Tanker (hazmat C)',
    84: 'Tanker (hazmat D)',
    89: 'Tanker (other)',
    90: 'Other vessel type',
    99: 'Other vessel type',
}

NAV_STATUS_MAP = {
    0:  'Under way using engine',
    1:  'At anchor',
    2:  'Not under command',
    3:  'Restricted manoeuvrability',
    4:  'Constrained by draught',
    5:  'Moored',
    6:  'Aground',
    7:  'Engaged in fishing',
    8:  'Under way sailing',
    9:  'Reserved (HSC)',
    10: 'Reserved (WIG)',
    11: 'Power-driven vessel towing astern',
    12: 'Power-driven vessel pushing ahead',
    13: 'Reserved',
    14: 'AIS SART active',
    15: 'Not defined',
}

def decode_vessel_type(code):
    if pd.isna(code):
        return 'Unknown'
    code_int = int(float(code))
    return VESSEL_TYPE_MAP.get(code_int, f'Type {code_int}')

def decode_nav_status(code):
    if pd.isna(code):
        return 'Not reported'
    code_int = int(float(code))
    return NAV_STATUS_MAP.get(code_int, f'Status {code_int}')

def speed_category(sog):
    if pd.isna(sog):
        return 'Unknown'
    if sog < 0.3:
        return 'Stationary'
    elif sog < 3.0:
        return 'Slow / manoeuvring'
    elif sog < 12:
        return 'Transit'
    else:
        return 'Cruising'

def vessel_category(type_name):
    t = str(type_name).lower()
    if 'tanker' in t:
        return 'Tanker'
    if 'cargo' in t:
        return 'Cargo'
    if 'tug' in t:
        return 'Tug / Support'
    if 'fishing' in t:
        return 'Fishing'
    if 'passenger' in t:
        return 'Passenger'
    if 'pilot' in t or 'sar' in t:
        return 'Pilot / SAR'
    if 'unknown' in t:
        return 'Unknown'
    return 'Other'

df['vessel_type_name'] = df['vessel_type'].apply(decode_vessel_type)
df['nav_status_name'] = df['navigational_status'].apply(decode_nav_status)
df['speed_category'] = df['sog_knots'].apply(speed_category)
df['is_moving'] = df['sog_knots'].apply(lambda x: False if pd.isna(x) else x > 0.3)
df['vessel_category'] = df['vessel_type_name'].apply(vessel_category)

print('Added new columns:')
print(['vessel_type_name', 'nav_status_name', 'speed_category', 'is_moving', 'vessel_category'])

print('\nVessel type name distribution:')
display(df['vessel_type_name'].value_counts(dropna=False).to_frame('count').head(15))

print('\nVessel category distribution:')
display(df['vessel_category'].value_counts(dropna=False).to_frame('count'))

Added new columns:
['vessel_type_name', 'nav_status_name', 'speed_category', 'is_moving', 'vessel_category']

Vessel type name distribution:


,count
vessel_type_name,
Unknown,797
Tanker,52
Cargo vessel,22
Tug,12
Vessel (anti-pollution),8
Cargo vessel (hazmat A),7
Other vessel type,6
Tanker (other),6
Cargo vessel (other),5



Vessel category distribution:


,count
vessel_category,
Unknown,797
Tanker,70
Cargo,38
Other,24
Tug / Support,12
Pilot / SAR,5
Fishing,3


## 7) Quick validation


In [12]:
print('Missing values by column:')
display(df.isna().sum().sort_values(ascending=False))

print('\nMessage type distribution:')
if 'message_type' in df.columns:
    display(df['message_type'].value_counts(dropna=False).to_frame('count'))

print('\nPort guess distribution:')
if 'port_guess' in df.columns:
    display(df['port_guess'].fillna('UNKNOWN').value_counts(dropna=False).to_frame('count'))


Missing values by column:


,0
source_timestamp,949
destination,815
imo,815
call_sign,815
vessel_type,797
navigational_status,246
true_heading,190
sog_knots,190
cog_degrees,190
ais_timestamp_field,173



Message type distribution:


,count
message_type,
PositionReport,703
ShipStaticData,134
StandardClassBPositionReport,55
StaticDataReport,27
AidsToNavigationReport,17
UnknownMessage,5
BaseStationReport,4
AddressedBinaryMessage,1
DataLinkManagementMessage,1



Port guess distribution:


,count
port_guess,
Singapore,949


## 8) Save cleaned output


In [13]:
# ── Save cleaned AIS data ──────────────────────────────────────────────
# Self-healing: if decode columns are missing (e.g. cell 17 wasn't run),
# apply the decoding inline before saving so the CSV is always complete.

VESSEL_TYPE_MAP = {
    0:'Not available',9:'Helicopter',13:'SAR aircraft',24:'Pilot vessel',
    25:'SAR vessel',26:'Tug',27:'Port tender',28:'Vessel (anti-pollution)',
    29:'Law enforcement',30:'Fishing vessel',31:'Towing vessel',
    32:'Towing (long/wide)',33:'Dredger',34:'Dive ops vessel',
    35:'Military vessel',36:'Sailing vessel',37:'Pleasure craft',
    40:'High-speed craft',41:'High-speed craft',50:'Pilot vessel',
    51:'SAR vessel',52:'Tug',53:'Port tender',54:'Anti-pollution vessel',
    55:'Law enforcement',57:'Medical transport',58:'Non-combatant ship',
    59:'Passenger ferry',60:'Passenger ship',61:'Passenger ship',
    62:'Passenger ship',63:'Passenger ship',69:'Passenger ship (other)',
    70:'Cargo vessel',71:'Cargo vessel (hazmat A)',72:'Cargo vessel (hazmat B)',
    73:'Cargo vessel (hazmat C)',74:'Cargo vessel (hazmat D)',
    77:'Cargo vessel',79:'Cargo vessel (other)',80:'Tanker',
    81:'Tanker (hazmat A)',82:'Tanker (hazmat B)',83:'Tanker (hazmat C)',
    84:'Tanker (hazmat D)',89:'Tanker (other)',90:'Other vessel type',99:'Other vessel type'
}
NAV_STATUS_MAP = {
    0:'Under way using engine',1:'At anchor',2:'Not under command',
    3:'Restricted manoeuvrability',4:'Constrained by draught',5:'Moored',
    6:'Aground',7:'Engaged in fishing',8:'Under way sailing',
    9:'Reserved (HSC)',10:'Reserved (WIG)',11:'Power-driven vessel towing astern',
    12:'Power-driven vessel pushing ahead',13:'Reserved',
    14:'AIS SART active',15:'Not defined'
}

def _decode_vessel(code):
    if pd.isna(code): return 'Unknown'
    return VESSEL_TYPE_MAP.get(int(float(code)), f'Type {int(float(code))}')

def _decode_nav(code):
    if pd.isna(code): return 'Not reported'
    return NAV_STATUS_MAP.get(int(float(code)), f'Status {int(float(code))}')

def _speed_cat(sog):
    if pd.isna(sog): return 'Unknown'
    if sog < 0.3: return 'Stationary'
    elif sog < 3.0: return 'Slow / manoeuvring'
    elif sog < 12: return 'Transit'
    return 'Cruising'

def _vessel_cat(t):
    t = str(t).lower()
    if 'tanker' in t: return 'Tanker'
    if 'cargo' in t: return 'Cargo'
    if 'tug' in t: return 'Tug / Support'
    if 'fishing' in t: return 'Fishing'
    if 'passenger' in t: return 'Passenger'
    if 'pilot' in t or 'sar' in t: return 'Pilot / SAR'
    if 'unknown' in t: return 'Unknown'
    return 'Other'

# Apply decode if not already done
if 'vessel_type_name' not in df.columns:
    if 'vessel_type' in df.columns:
        df['vessel_type'] = pd.to_numeric(df['vessel_type'], errors='coerce')
    if 'navigational_status' in df.columns:
        df['navigational_status'] = pd.to_numeric(df['navigational_status'], errors='coerce')
    df['vessel_type_name'] = df['vessel_type'].apply(_decode_vessel)
    df['nav_status_name']  = df['navigational_status'].apply(_decode_nav)
    df['speed_category']   = df['sog_knots'].apply(_speed_cat)
    df['is_moving']        = df['sog_knots'].apply(lambda x: False if pd.isna(x) else x > 0.3)
    df['vessel_category']  = df['vessel_type_name'].apply(_vessel_cat)
    print('Decode applied inline (cell 17 result not found in df)')
else:
    print('Decode columns already present from cell 17')

final_cols = [
    'event_time_utc', 'capture_utc', 'message_type',
    'mmsi', 'ship_name', 'imo', 'call_sign',
    'latitude', 'longitude',
    'sog_knots', 'speed_category', 'is_moving',
    'cog_degrees', 'true_heading',
    'navigational_status', 'nav_status_name',
    'destination', 'eta',
    'vessel_type', 'vessel_type_name', 'vessel_category',
    'port_guess', 'source', 'data_type', 'granularity',
    'year', 'month', 'day', 'hour'
]
final_cols = [c for c in final_cols if c in df.columns]
df = df[final_cols].copy()

df.to_csv(CLEAN_CSV_PATH, index=False)
print(f'Clean CSV saved → {CLEAN_CSV_PATH}')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'New columns: vessel_type_name, nav_status_name, speed_category, is_moving, vessel_category')
print()
print('vessel_category distribution:')
print(df['vessel_category'].value_counts().to_string())
print()
print('speed_category distribution:')
print(df['speed_category'].value_counts().to_string())
df.head(5)

Decode columns already present from cell 17
Clean CSV saved → /content/drive/MyDrive/repo/data/clean/aisstream_clean.csv
Shape: 949 rows x 29 columns
New columns: vessel_type_name, nav_status_name, speed_category, is_moving, vessel_category

vessel_category distribution:
vessel_category
Unknown          797
Tanker            70
Cargo             38
Other             24
Tug / Support     12
Pilot / SAR        5
Fishing            3

speed_category distribution:
speed_category
Stationary            471
Transit               207
Unknown               190
Cruising               49
Slow / manoeuvring     32


,event_time_utc,capture_utc,message_type,mmsi,ship_name,imo,call_sign,latitude,longitude,sog_knots,speed_category,is_moving,cog_degrees,true_heading,navigational_status,nav_status_name,destination,eta,vessel_type,vessel_type_name,vessel_category,port_guess,source,data_type,granularity,year,month,day,hour
0,2026-04-04 21:16:19.891862+00:00,2026-04-04 21:16:19.891862+00:00,StaticDataReport,525100095,MEGAMAS SWEET,NaN,<NA>,1.26393,103.83230,NaN,Unknown,False,NaN,NaN,NaN,Not reported,<NA>,None,NaN,Unknown,Unknown,Singapore,aisstream,vessel_movement,event,2026,4,4,21
1,2026-04-04 21:16:19.891862+00:00,2026-04-04 21:16:19.891862+00:00,PositionReport,636013118,YM INSTRUCTION,NaN,<NA>,1.21927,103.81743,1.8,Slow / manoeuvring,True,240.5,270.0,0.0,Under way using engine,<NA>,None,NaN,Unknown,Unknown,Singapore,aisstream,vessel_movement,event,2026,4,4,21
2,2026-04-04 21:16:19.891862+00:00,2026-04-04 21:16:19.891862+00:00,PositionReport,566753000,HAI SOON 33,NaN,<NA>,1.20711,103.85619,7.9,Transit,True,64.6,52.0,0.0,Under way using engine,<NA>,None,NaN,Unknown,Unknown,Singapore,aisstream,vessel_movement,event,2026,4,4,21
3,2026-04-04 21:16:19.891862+00:00,2026-04-04 21:16:19.891862+00:00,PositionReport,538011746,BLUE GRETHA,NaN,<NA>,1.22248,103.90738,5.0,Transit,True,248.1,247.0,0.0,Under way using engine,<NA>,None,NaN,Unknown,Unknown,Singapore,aisstream,vessel_movement,event,2026,4,4,21
4,2026-04-04 21:16:19.891862+00:00,2026-04-04 21:16:19.891862+00:00,PositionReport,563074570,VISION 225,NaN,<NA>,1.26290,103.82865,0.0,Stationary,False,228.6,511.0,15.0,Not defined,<NA>,None,NaN,Unknown,Unknown,Singapore,aisstream,vessel_movement,event,2026,4,4,21


## 9) Optional quick summary for the report


In [14]:
summary = {
    'rows_captured': int(len(df)),
    'unique_mmsi': int(df['mmsi'].nunique(dropna=True)) if 'mmsi' in df.columns else None,
    'message_types': df['message_type'].dropna().unique().tolist()[:10] if 'message_type' in df.columns else [],
    'ports_detected': df['port_guess'].dropna().unique().tolist()[:20] if 'port_guess' in df.columns else [],
    'time_min_utc': str(df['event_time_utc'].min()) if 'event_time_utc' in df.columns and len(df) else None,
    'time_max_utc': str(df['event_time_utc'].max()) if 'event_time_utc' in df.columns and len(df) else None,
}
summary


{'rows_captured': 949,
 'unique_mmsi': 419,
 'message_types': ['StaticDataReport',
  'PositionReport',
  'ShipStaticData',
  'AddressedBinaryMessage',
  'StandardClassBPositionReport',
  'BaseStationReport',
  'AidsToNavigationReport',
  'DataLinkManagementMessage',
  'UnknownMessage',
  'ExtendedClassBPositionReport'],
 'ports_detected': ['Singapore'],
 'time_min_utc': '2026-04-04 21:16:19.891862+00:00',
 'time_max_utc': '2026-04-04 21:16:19.891862+00:00'}